# 13 · Events (thin detection layer)

Derives **`monitoring.events`** — one governed, cross-twin table of abnormalities the control tower and (later) the agent act on. Each row is a detected breach in a twin: stockout / low cover / overstock, supplier risk / low OTIF, bottleneck / low yield, fulfillment risk / low fill, forecast drift.

**Thin by design:** it *detects* (from the `*_twin_current` tables) and carries an empty `recommended_action` for the planning engine to fill later. Rebuilt deterministically each run (idempotent overwrite); mutable status/ownership workflow moves to Lakebase in a later step.

Thresholds are config at the top — tune per business.

## Config

In [ ]:
import uuid
dbutils.widgets.text("target_catalog", "mickmickstableawscoff_catalog", "Target catalog")
dbutils.widgets.text("monitoring_schema", "monitoring", "Monitoring schema")
FQ_MON = f"{dbutils.widgets.get('target_catalog').strip()}.{dbutils.widgets.get('monitoring_schema').strip()}"

def cfg(key, default, kind="double"):
    """Effective GLOBAL planning_config value with fallback (runnable even before 03_planning_config)."""
    col = "value_string" if kind == "string" else "value_double"
    try:
        r = spark.sql(f"SELECT {col} FROM {FQ_MON}.v_resolved_config WHERE param_key = '{key}'").collect()
        return r[0][0] if r and r[0][0] is not None else default
    except Exception:
        return default

REFRESH_ID = uuid.uuid4().hex

# --- thresholds (TODO — EXTEND for your business) ---
LOW_OTIF = cfg("procurement_risk_otif", 0.80)    # OTIF below this raises LOW_OTIF
LOW_YIELD = cfg("low_yield_pct", 90.0)           # yield %% below this raises LOW_YIELD
LOW_FILL = cfg("low_fill_rate", 0.95)            # fill rate below this raises LOW_FILL
DRIFT_WMAPE = cfg("forecast_drift_wmape", 0.60)  # WMAPE above this raises FORECAST_DRIFT
print("events target:", f"{FQ_MON}.events")

## Build events
Each twin contributes typed events with a common 14-column shape; the outer query adds a deterministic `event_id`, default `owner`/`status`, and an empty `recommended_action`.

In [ ]:
inner = f"""
-- INVENTORY -------------------------------------------------------------
SELECT 'STOCKOUT' AS event_type, 'INVENTORY' AS function, 'HIGH' AS severity, 'MATERIAL' AS entity_type,
  material AS entity_id, material_text AS entity_label, plant, CAST(NULL AS STRING) AS channel,
  concat('Stockout: ', material_text, ' @ plant ', plant) AS title,
  concat('on_hand=', CAST(round(on_hand_qty,0) AS STRING), ', days_cover=', CAST(round(days_of_cover_effective,1) AS STRING)) AS detail,
  CAST(days_of_cover_effective AS DOUBLE) AS metric_value, CAST(0.0 AS DOUBLE) AS threshold_value,
  'inventory_twin_current' AS source_table, as_of_date
FROM {FQ_MON}.inventory_twin_current WHERE is_stockout
UNION ALL
SELECT 'LOW_COVER','INVENTORY','MEDIUM','MATERIAL', material, material_text, plant, CAST(NULL AS STRING),
  concat('Low cover: ', material_text, ' @ plant ', plant),
  concat('days_cover=', CAST(round(days_of_cover_effective,1) AS STRING)),
  CAST(days_of_cover_effective AS DOUBLE), CAST(NULL AS DOUBLE), 'inventory_twin_current', as_of_date
FROM {FQ_MON}.inventory_twin_current WHERE is_low_cover AND NOT is_stockout
UNION ALL
SELECT 'OVERSTOCK','INVENTORY','LOW','MATERIAL', material, material_text, plant, CAST(NULL AS STRING),
  concat('Overstock: ', material_text, ' @ plant ', plant),
  concat('on_hand=', CAST(round(on_hand_qty,0) AS STRING), ', days_cover=', CAST(round(days_of_cover_effective,1) AS STRING)),
  CAST(days_of_cover_effective AS DOUBLE), CAST(NULL AS DOUBLE), 'inventory_twin_current', as_of_date
FROM {FQ_MON}.inventory_twin_current WHERE is_overstock
UNION ALL
-- PROCUREMENT -----------------------------------------------------------
SELECT 'SUPPLIER_RISK','PROCUREMENT','HIGH','SUPPLIER', supplier, supplier_name, plant, CAST(NULL AS STRING),
  concat('Supplier at risk: ', supplier_name, ' (', material_text, ')'),
  concat('otif=', CAST(round(otif_rate,2) AS STRING), ', lead_days=', CAST(round(avg_lead_time_days,1) AS STRING)),
  CAST(otif_rate AS DOUBLE), CAST(NULL AS DOUBLE), 'procurement_twin_current', as_of_date
FROM {FQ_MON}.procurement_twin_current WHERE supplier_risk_flag
UNION ALL
SELECT 'LOW_OTIF','PROCUREMENT','MEDIUM','SUPPLIER', supplier, supplier_name, plant, CAST(NULL AS STRING),
  concat('Low OTIF: ', supplier_name, ' (', material_text, ')'),
  concat('otif=', CAST(round(otif_rate,2) AS STRING)),
  CAST(otif_rate AS DOUBLE), CAST({LOW_OTIF} AS DOUBLE), 'procurement_twin_current', as_of_date
FROM {FQ_MON}.procurement_twin_current WHERE otif_rate < {LOW_OTIF} AND NOT supplier_risk_flag
UNION ALL
-- PRODUCTION ------------------------------------------------------------
SELECT 'BOTTLENECK','PRODUCTION','MEDIUM','MATERIAL', material, material_text, plant, CAST(NULL AS STRING),
  concat('Bottleneck: ', material_text, ' @ ', primary_work_centre),
  concat('open_prod=', CAST(round(open_production_qty,0) AS STRING), ', work_centre=', primary_work_centre),
  CAST(open_production_qty AS DOUBLE), CAST(NULL AS DOUBLE), 'production_twin_current', as_of_date
FROM {FQ_MON}.production_twin_current WHERE is_bottleneck
UNION ALL
SELECT 'LOW_YIELD','PRODUCTION','MEDIUM','MATERIAL', material, material_text, plant, CAST(NULL AS STRING),
  concat('Low yield: ', material_text, ' @ plant ', plant),
  concat('yield_pct=', CAST(round(yield_pct,1) AS STRING)),
  CAST(yield_pct AS DOUBLE), CAST({LOW_YIELD} AS DOUBLE), 'production_twin_current', as_of_date
FROM {FQ_MON}.production_twin_current WHERE yield_pct < {LOW_YIELD}
UNION ALL
-- DISTRIBUTION ----------------------------------------------------------
SELECT 'FULFILLMENT_RISK','DISTRIBUTION','HIGH','MATERIAL', material, material_text, plant, channel,
  concat('Fulfillment risk: ', material_text, ' (', channel, ')'),
  concat('fill=', CAST(round(fill_rate,2) AS STRING), ', otif=', CAST(round(otif_rate,2) AS STRING)),
  CAST(fill_rate AS DOUBLE), CAST(NULL AS DOUBLE), 'distribution_twin_current', as_of_date
FROM {FQ_MON}.distribution_twin_current WHERE fulfillment_risk_flag
UNION ALL
SELECT 'LOW_FILL','DISTRIBUTION','MEDIUM','MATERIAL', material, material_text, plant, channel,
  concat('Low fill rate: ', material_text, ' (', channel, ')'),
  concat('fill=', CAST(round(fill_rate,2) AS STRING)),
  CAST(fill_rate AS DOUBLE), CAST({LOW_FILL} AS DOUBLE), 'distribution_twin_current', as_of_date
FROM {FQ_MON}.distribution_twin_current WHERE fill_rate < {LOW_FILL} AND NOT fulfillment_risk_flag
UNION ALL
-- DEMAND ----------------------------------------------------------------
SELECT 'FORECAST_DRIFT','DEMAND','MEDIUM','MATERIAL', material, material_text, plant, channel,
  concat('High forecast error: ', material_text, ' (', channel, ')'),
  concat('wmape=', CAST(round(forecast_wmape,2) AS STRING)),
  CAST(forecast_wmape AS DOUBLE), CAST({DRIFT_WMAPE} AS DOUBLE), 'demand_twin_current', as_of_date
FROM {FQ_MON}.demand_twin_current WHERE forecast_wmape > {DRIFT_WMAPE}
"""

events = spark.sql(f"""
SELECT
  sha2(concat_ws('|', event_type, entity_id, coalesce(plant,''), coalesce(channel,'')), 256) AS event_id,
  event_type, function, severity, entity_type, entity_id, entity_label, plant, channel,
  title, detail, metric_value, threshold_value,
  CAST(NULL AS STRING) AS recommended_action,
  CASE function WHEN 'DEMAND' THEN 'Demand Planner' WHEN 'INVENTORY' THEN 'Inventory Planner'
    WHEN 'PROCUREMENT' THEN 'Procurement Manager' WHEN 'PRODUCTION' THEN 'Production Planner'
    WHEN 'DISTRIBUTION' THEN 'Logistics Manager' END AS owner,
  'open' AS status,
  source_table, as_of_date, current_timestamp() AS detected_at,
  '{REFRESH_ID}' AS refresh_id, current_timestamp() AS generated_at
FROM (
{inner}
)
""")

(events.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{FQ_MON}.events"))
spark.sql(f"ALTER TABLE {FQ_MON}.events SET TBLPROPERTIES ('comment' = 'Cross-twin detected events (abnormalities). Rebuilt deterministically from the *_twin_current tables each run; recommended_action is filled by the planning engine.')")
print("wrote", f"{FQ_MON}.events")

## Column descriptions

In [ ]:
COMMENTS = {
  "event_id": "Deterministic id = sha2(event_type|entity_id|plant|channel); stable across runs",
  "event_type": "STOCKOUT | LOW_COVER | OVERSTOCK | SUPPLIER_RISK | LOW_OTIF | BOTTLENECK | LOW_YIELD | FULFILLMENT_RISK | LOW_FILL | FORECAST_DRIFT",
  "function": "Owning twin/domain: DEMAND | INVENTORY | PROCUREMENT | PRODUCTION | DISTRIBUTION",
  "severity": "HIGH | MEDIUM | LOW",
  "entity_type": "Type of the affected entity (MATERIAL | SUPPLIER)",
  "entity_id": "Id of the affected entity (material number or supplier code)",
  "entity_label": "Human-readable label (material text or supplier name)",
  "plant": "Plant context (nullable)",
  "channel": "Distribution/demand channel context (nullable)",
  "title": "One-line human summary of the event",
  "detail": "Key metrics behind the event",
  "metric_value": "The measured value that triggered the event",
  "threshold_value": "The threshold breached (nullable where flag-driven)",
  "recommended_action": "Suggested remediation (filled later by the planning engine; NULL for now)",
  "owner": "Default responsible role for this function",
  "status": "Lifecycle status (open now; mutated in the plan-of-record store later)",
  "source_table": "Twin table the event was derived from",
  "as_of_date": "As-of date of the source twin row",
  "detected_at": "Timestamp this event was derived",
  "refresh_id": "Run id of this events build",
  "generated_at": "Build timestamp",
}
for col, c in COMMENTS.items():
    spark.sql(f"ALTER TABLE {FQ_MON}.events ALTER COLUMN {col} COMMENT '" + c.replace("'", "''") + "'")
print("column comments set:", len(COMMENTS))

## Verify

In [ ]:
print("=== events by function / severity ===")
display(spark.sql(f"SELECT function, severity, COUNT(*) n FROM {FQ_MON}.events GROUP BY function, severity ORDER BY function, severity"))
print("=== events by type ===")
display(spark.sql(f"SELECT event_type, COUNT(*) n FROM {FQ_MON}.events GROUP BY event_type ORDER BY n DESC"))
print("=== sample HIGH/MEDIUM events ===")
display(spark.sql(f"SELECT severity, function, event_type, entity_label, plant, channel, title, detail FROM {FQ_MON}.events ORDER BY CASE severity WHEN 'HIGH' THEN 0 WHEN 'MEDIUM' THEN 1 ELSE 2 END LIMIT 25"))

## Next
Events populated. Next: the **planning engine** (inventory policy + MRP — the unblocked parts) writes `recommended_action` back onto open events. Later, an **Events** tab in the Twin Explorer app and a mutable status workflow in Lakebase.